In [ ]:
!pip install -U "tensorflow==2.19.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 80.4 MB/s eta 0:00:00
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.20.0
    Uninstalling tensorboard-2.20.0:
      Successfully uninstalled tensorboard-2.20.0
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.20.0
    Uninstalling tensorflow-2.20.0:
      Successfully uninstalled tensorflow-2.20.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.19.1 which is incompatible.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import tensorflow as tf
import os

base_dir = "/content/drive/MyDrive/Proyecto IA/Data"

In [4]:
model = tf.keras.models.load_model(os.path.join(base_dir,'feeling_model_2.keras'))

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,143,694 (8.18 MB)

 Trainable params: 714,564 (2.73 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,429,130 (5.45 MB)

In [5]:
test_dataset = tf.data.Dataset.load(os.path.join(base_dir, "test_dataset"))

In [6]:
test_loss, test_acc = model.evaluate(test_dataset)

print('Test Loss:', test_loss)
print('Test Accuracy:', test_acc)

431/431 ━━━━━━━━━━━━━━━━━━━━ 30s 67ms/step - accuracy: 0.9723 - loss: 0.0689
Test Loss: 0.06868446618318558
Test Accuracy: 0.9713030457496643


In [7]:
class_names = ["sadness", "joy", "anger", "fear"]
class_to_index = {name: i for i, name in enumerate(class_names)}
index_to_class = {i: name for i, name in enumerate(class_names)}
print(class_to_index)

{'sadness': 0, 'joy': 1, 'anger': 2, 'fear': 3}


In [8]:
with open(os.path.join(base_dir,'vocab.txt'), 'r', encoding='utf-8') as f:
    vocab = [line.strip() for line in f if line.strip()]

In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=len(vocab))
tokenizer.word_index = {word: i+1 for i, word in enumerate(vocab)}

In [10]:
sample_text = ["I have an important exam and I feel nervous"]
seq = tokenizer.texts_to_sequences(sample_text)

seq

[[2, 18, 75, 280, 1632, 4, 2, 3, 331]]

In [11]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

MAX_LEN = 100
x = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')

pred = model.predict(x)
print(f"Array: {pred}")

predicted_index = np.argmax(pred)
predicted_label = index_to_class[predicted_index]

print(f"Predicción: {predicted_label}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 655ms/step
Array: [[5.77452534e-04 1.50558135e-05 1.41297744e-06 9.99406099e-01]]
Predicción: fear


In [12]:
sample_text = ["I feel frustrated with the project"]
seq = tokenizer.texts_to_sequences(sample_text)

seq

[[2, 3, 298, 25, 6, 973]]

In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

MAX_LEN = 100
x = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')

pred = model.predict(x)
print(f"Array: {pred}")

predicted_index = np.argmax(pred)
predicted_label = index_to_class[predicted_index]

print(f"Predicción: {predicted_label}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
Array: [[3.5336114e-07 2.0927091e-07 9.9999297e-01 6.4633605e-06]]
Predicción: anger


In [18]:
texts = [
    # anger
    "I feel frustrated with the project",
    "This is infuriating, nothing works as expected",

    # joy
    "I’m really happy with our progress",
    "Today was amazing, everything clicked",

    # sadness
    "I feel disappointed by the outcome",
    "Today I just feel empty and tired",

    # fear
    "I’m worried we might miss the deadline",
    "I feel anxious about presenting tomorrow",
]


In [19]:
def load_tokenizer_from_vocab(vocab_path):
    with open(vocab_path, 'r', encoding='utf-8') as f:
        vocab = [line.strip() for line in f if line.strip()]
    # Índices 1..N (0 se usa para padding)
    word_index = {word: i+1 for i, word in enumerate(vocab)}
    tok = Tokenizer(num_words=len(vocab))
    tok.word_index = word_index
    return tok

def vectorize_texts(texts, tokenizer, maxlen):
    seqs = tokenizer.texts_to_sequences(texts)
    x = pad_sequences(seqs, maxlen=maxlen, padding='post', truncating='post')
    return x

In [20]:
import pandas as pd

X = vectorize_texts(texts, tokenizer, MAX_LEN)
probs = model.predict(X)  # shape: (n_texts, 6)

# === Construir DataFrame con probabilidades y top-1 ===
df = pd.DataFrame(probs, columns=class_names)
df.insert(0, "text", texts)

pred_idx = np.argmax(probs, axis=1)
pred_label = [index_to_class[i] for i in pred_idx]
confidence = probs[np.arange(len(probs)), pred_idx]

df["predicted"] = pred_label
df["confidence"] = confidence

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step


In [21]:
df

,text,sadness,joy,anger,fear,predicted,confidence
0,I feel frustrated with the project,3.533618e-07,2.092713e-07,9.999929e-01,6.463360e-06,anger,0.999993
1,"This is infuriating, nothing works as expected",1.005905e-05,3.604271e-03,4.936927e-02,9.470164e-01,fear,0.947016
2,I’m really happy with our progress,1.635930e-03,9.835390e-01,1.047417e-02,4.350950e-03,joy,0.983539
3,"Today was amazing, everything clicked",1.234138e-03,9.862754e-01,1.214543e-02,3.449385e-04,joy,0.986275
4,I feel disappointed by the outcome,9.999999e-01,1.859356e-10,7.584397e-10,4.259320e-12,sadness,1.000000
5,Today I just feel empty and tired,9.999967e-01,3.164711e-06,1.891683e-08,1.018551e-09,sadness,0.999997
6,I’m worried we might miss the deadline,4.135179e-01,7.941324e-02,4.247754e-01,8.229350e-02,anger,0.424775
7,I feel anxious about presenting tomorrow,9.215233e-06,2.171093e-08,5.524936e-06,9.999852e-01,fear,0.999985
